# Knowledge Distillation — Paper-to-Code Mock (Colab)

**Paper:** Distilling the Knowledge in a Neural Network (Hinton, Vinyals & Dean, 2015) — https://arxiv.org/abs/1503.02531

Medium mock (~30 min). Fill in the `kd_loss` stub, run the teacher/student demo, then the sanity checks. Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

N_CLASSES = 4

## 1. Implement the distillation loss (YOUR TASK)

Soft term: `T^2 * KL( softmax(teacher/T) || softmax(student/T) )`. Optional hard term: cross-entropy with labels, weighted by `alpha`. The teacher must be FROZEN (detach its logits).

In [ ]:
def kd_loss(student_logits, teacher_logits, targets=None, T=4.0, alpha=0.1):
    # TODO: detach teacher_logits (freeze the teacher)
    # TODO: log_p_student = log_softmax(student_logits / T)
    # TODO: p_teacher = softmax(teacher_logits / T)
    # TODO: soft = kl_div(log_p_student, p_teacher, reduction='batchmean') * T * T
    # TODO: if targets is None or alpha == 0: return soft
    # TODO: hard = cross_entropy(student_logits, targets)
    # TODO: return (1 - alpha) * soft + alpha * hard
    raise NotImplementedError("fill me in")

## 2. Demonstrate the benefit (dark knowledge survives label noise)

A big teacher learns the true structure from CLEAN labels. A tiny student is trained on NOISY labels (40% flipped). The student that also matches the frozen teacher's soft targets recovers the truth and generalizes better.

In [ ]:
def make_blobs(n_per_class, seed, spread=0.8):
    g = torch.Generator().manual_seed(seed)
    angles = torch.arange(N_CLASSES) * (2 * 3.14159265 / N_CLASSES)
    centers = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * 2.5
    X = [centers[c] + spread * torch.randn(n_per_class, 2, generator=g) for c in range(N_CLASSES)]
    y = [torch.full((n_per_class,), c, dtype=torch.long) for c in range(N_CLASSES)]
    return torch.cat(X), torch.cat(y)

def corrupt_labels(y, frac, seed):
    g = torch.Generator().manual_seed(seed); yc = y.clone()
    flip = torch.rand(len(y), generator=g) < frac
    yc[flip] = torch.randint(0, N_CLASSES, (int(flip.sum()),), generator=g)
    return yc

def make_net(hidden):
    return nn.Sequential(nn.Linear(2, hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden), nn.ReLU(),
                         nn.Linear(hidden, N_CLASSES))

def accuracy(net, X, y):
    net.eval()
    with torch.no_grad():
        return (net(X).argmax(-1) == y).float().mean().item()

def train_teacher(Xtr, ytr, seed):
    torch.manual_seed(seed); net = make_net(64)
    opt = torch.optim.Adam(net.parameters(), lr=1e-2); net.train()
    for _ in range(500):
        loss = F.cross_entropy(net(Xtr), ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    return net

def train_student(Xtr, ytr_noisy, teacher, seed, use_kd, T=4.0, alpha=0.1):
    torch.manual_seed(seed); net = make_net(8)
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    tlog = None
    if teacher is not None:
        teacher.eval()
        with torch.no_grad():
            tlog = teacher(Xtr)
    net.train()
    for _ in range(400):
        slog = net(Xtr)
        loss = kd_loss(slog, tlog, ytr_noisy, T=T, alpha=alpha) if use_kd \
               else F.cross_entropy(slog, ytr_noisy)
        opt.zero_grad(); loss.backward(); opt.step()
    return net

seeds = [0, 1, 2, 3, 4]
a_accs, k_accs = [], []
for s in seeds:
    Xtr, ytr = make_blobs(60, seed=s)
    Xte, yte = make_blobs(400, seed=s + 999)
    teacher = train_teacher(Xtr, ytr, seed=s)
    ytr_noisy = corrupt_labels(ytr, frac=0.4, seed=s + 7)
    alone = train_student(Xtr, ytr_noisy, None, seed=s + 1, use_kd=False)
    kd    = train_student(Xtr, ytr_noisy, teacher, seed=s + 1, use_kd=True)
    a_accs.append(accuracy(alone, Xte, yte))
    k_accs.append(accuracy(kd, Xte, yte))

mean = lambda xs: sum(xs) / len(xs)
print(f"student-alone test acc : {mean(a_accs):.3f}")
print(f"student+KD   test acc : {mean(k_accs):.3f}")

## 3. Sanity checks

In [ ]:
torch.manual_seed(7)
B = 16
student_logits = torch.randn(B, N_CLASSES, requires_grad=True)
teacher_logits = torch.randn(B, N_CLASSES)
targets = torch.randint(0, N_CLASSES, (B,))

# 1) T=1, alpha=1 reduces to CE; alpha=0,T=1 reduces to plain KL
assert torch.allclose(kd_loss(student_logits, teacher_logits, targets, T=1.0, alpha=1.0),
                      F.cross_entropy(student_logits, targets))
assert torch.allclose(kd_loss(student_logits, teacher_logits, T=1.0, alpha=0.0),
                      F.kl_div(F.log_softmax(student_logits, -1), F.softmax(teacher_logits, -1),
                               reduction='batchmean'))

# 2) softened teacher targets sum to 1 and are softer (higher entropy) at higher T
def soft_stats(T):
    p = F.softmax(teacher_logits / T, dim=-1)
    return p.sum(-1), -(p * p.clamp_min(1e-12).log()).sum(-1).mean()
s_lo, e_lo = soft_stats(1.0); s_hi, e_hi = soft_stats(8.0)
assert torch.allclose(s_lo, torch.ones(B)) and torch.allclose(s_hi, torch.ones(B))
assert e_hi > e_lo

# 3) teacher receives no gradient (frozen)
teacher = train_teacher(*make_blobs(20, seed=0), seed=0)
for p in teacher.parameters(): p.grad = None
student = make_net(8); Xs, _ = make_blobs(8, seed=5); teacher.eval()
with torch.no_grad(): tlog = teacher(Xs)
kd_loss(student(Xs), tlog, T=4.0, alpha=0.0).backward()
assert all(p.grad is None for p in teacher.parameters())

# 4) gradient flows to the student
assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in student.parameters())

# 5) student+KD >= student-alone (the demonstration)
assert mean(k_accs) >= mean(a_accs)

# 6) KD soft loss ~0 when student logits == teacher logits
sl = torch.randn(B, N_CLASSES)
assert kd_loss(sl.clone().requires_grad_(True), sl.clone(), T=4.0, alpha=0.0).item() < 1e-6

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
def kd_loss_solution(student_logits, teacher_logits, targets=None, T=4.0, alpha=0.1):
    teacher_logits = teacher_logits.detach()              # freeze the teacher
    log_p_student = F.log_softmax(student_logits / T, dim=-1)
    p_teacher = F.softmax(teacher_logits / T, dim=-1)
    soft = F.kl_div(log_p_student, p_teacher, reduction='batchmean') * (T * T)
    if targets is None or alpha == 0.0:
        return soft
    hard = F.cross_entropy(student_logits, targets)
    return (1.0 - alpha) * soft + alpha * hard